In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [3]:
import json
import os
from transformers import AutoTokenizer

# ========== 配置 ==========
MODEL_DIR = "/root/autodl-tmp/flan-t5-xl"   # 请确认实际路径
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# 数据集名称及对应的文件路径（请根据实际存放位置修改）
datasets = {
    "trivia": {
        "sarrag": "trivia/final_results.jsonl",                # SaR-RAG 预测文件
        "onestep": "trivia/trivia_bm25_top10_onestep_results.jsonl"  # One-step 预测文件
    },
    "hotpot": {
        "sarrag": "hotpot/final_results.jsonl",
        "onestep": "hotpot/hotpot_bm25_top10_onestep_results.jsonl"
    },
    "nq": {
        "sarrag": "nq/final_results.jsonl",
        "onestep": "nq/nq_bm25_top10_onestep_results.jsonl"
    }
}

def compute_avg_token_length(file_path, pred_field="prediction"):
    """计算 jsonl 文件中 pred_field 字段的平均 token 长度"""
    if not os.path.exists(file_path):
        print(f"警告：文件不存在 {file_path}")
        return None, 0, 0
    
    total_tokens = 0
    num_samples = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            pred_text = data.get(pred_field, "")
            # 使用 tokenizer 编码（不添加特殊标记，直接计数）
            tokens = tokenizer.encode(pred_text, add_special_tokens=False)
            total_tokens += len(tokens)
            num_samples += 1
    
    if num_samples == 0:
        return 0.0, 0, 0
    avg = total_tokens / num_samples
    return avg, total_tokens, num_samples

# ========== 主程序 ==========
print("="*50)
print("输出 token 平均长度统计 (Flan-T5-XL tokenizer)")
print("="*50)

for ds_name, paths in datasets.items():
    print(f"\n【{ds_name.upper()}】")
    
    # SaR-RAG
    avg_sar, total_sar, cnt_sar = compute_avg_token_length(paths["sarrag"])
    if avg_sar is not None:
        print(f"  SaR-RAG   : 平均 {avg_sar:.2f} tokens (总计 {total_sar}, 样本数 {cnt_sar})")
    
    # One-step
    avg_one, total_one, cnt_one = compute_avg_token_length(paths["onestep"])
    if avg_one is not None:
        print(f"  One-step  : 平均 {avg_one:.2f} tokens (总计 {total_one}, 样本数 {cnt_one})")

print("\n完成。")

输出 token 平均长度统计 (Flan-T5-XL tokenizer)

【TRIVIA】
  SaR-RAG   : 平均 3.79 tokens (总计 1895, 样本数 500)
  One-step  : 平均 3.37 tokens (总计 1687, 样本数 500)

【HOTPOT】
  SaR-RAG   : 平均 3.44 tokens (总计 1722, 样本数 500)
  One-step  : 平均 3.36 tokens (总计 1679, 样本数 500)

【NQ】
  SaR-RAG   : 平均 4.48 tokens (总计 2242, 样本数 500)
  One-step  : 平均 4.99 tokens (总计 2496, 样本数 500)

完成。
